### Vector DataBase

Vector Store: Small Scale and Mostly Local

Vector DB: Large and Online Based

In [140]:
from dotenv import load_dotenv
load_dotenv()

True

In [141]:
# Load HuggingFace Token
import os
os.environ['HF_TOKEN']=os.getenv("HF_TOKEN")

In [142]:
from langchain_huggingface import HuggingFaceEmbeddings
embedding_model=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Embedding Model feature representation = 384
len(embedding_model.embed_query("hello AI"))

384

#### Types of Similarity Scores

Lets Check Similarity between Data and Query

| Metric            | Similarity Score Range | Behavior                              |
| ----------------- | ---------------------- | ------------------------------------- |
| Cosine Similarity | \[-1, 1]               | Focuses on angle only |
| L2 Distance       | \[0, ∞)                | Focuses on **magnitude + direction**  |


In [143]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics.pairwise import euclidean_distances #L2

In [144]:
# Lets Embed Documents
documents=["what is a capital of USA?",
           "Who is a president of USA?",
           "Who is a prime minister of India?"]

document_embedding=embedding_model.embed_documents(documents)

In [145]:
# Lets Embed Query
my_query="Narendra modi is prime minister of india?" # More Similar to 3rd Document

query_embedding=embedding_model.embed_query(my_query)

Lets Check Similarity Score: Cosine Similarity vs L2 (Euclidean Distance)

In [146]:
# Cosine Similarity Result
cosine_similarity([query_embedding],document_embedding)

array([[0.11756677, 0.34324558, 0.81413233]])

In [147]:
# L2 Similarity Results
euclidean_distances([query_embedding], document_embedding)

array([[1.32848279, 1.14608414, 0.60970103]])

#### FAISS

Facebook AI Similarity Search

Vector Store

In [148]:
import faiss
from langchain_community.vectorstores import FAISS

# InMemoryDocstore stores documents in memory for quick retrieval by ID.
from langchain_community.docstore.in_memory import InMemoryDocstore

#### Index

Whenever we store documents, we need to create an index. This index helps in quickly finding similar documents.

In [149]:
# Indexing the Documents with FAISS, where L2 is Euclidean Distance
index=faiss.IndexFlatL2(384) #384 is dimension of embedding model`

In [150]:
# Defining FAISS
vector_store=FAISS(
    embedding_function=embedding_model,
    index=index,
    docstore=InMemoryDocstore(), #stores documents in memory for quick retrieval by ID
    index_to_docstore_id={}, #maps index positions to document IDs
)

In [151]:
# Add Docs in vector Store
vector_store.add_texts(["AI is future","AI is powerful","Dogs are cute"])

['dc3e04d4-f0f9-4d66-bcf9-b1d02115272a',
 '804bf852-6e4b-4097-b4cb-e7dc71369d81',
 '8a101d5d-f75e-49ea-980d-47f2ad156744']

In [152]:
# Add Index to docs
vector_store.index_to_docstore_id

{0: 'dc3e04d4-f0f9-4d66-bcf9-b1d02115272a',
 1: '804bf852-6e4b-4097-b4cb-e7dc71369d81',
 2: '8a101d5d-f75e-49ea-980d-47f2ad156744'}

In [153]:
# K will return the 3 closest documents to the query based on similarity (e.g., cosine or L2 distance).
results = vector_store.similarity_search("Tell me about AI", k=3)
results

[Document(id='804bf852-6e4b-4097-b4cb-e7dc71369d81', metadata={}, page_content='AI is powerful'),
 Document(id='dc3e04d4-f0f9-4d66-bcf9-b1d02115272a', metadata={}, page_content='AI is future'),
 Document(id='8a101d5d-f75e-49ea-980d-47f2ad156744', metadata={}, page_content='Dogs are cute')]

| Feature               | `Flat`                | `IVF` (Inverted File Index)        | `HNSW` (Graph-based Index)          |
| --------------------- | --------------------- | ---------------------------------- | ----------------------------------- |
| Type of Search     | Exact                 | Approximate (cluster-based)        | Approximate (graph-based traversal) |
| Speed               | Slow (linear scan)    | Fast (search only in top clusters) | Very Fast (graph walk)              |


| Dataset Size              | Recommended Index                 |
| ------------------------- | --------------------------------- |
| UPTO 1L                     | `IndexFlatL2` or `IndexFlatIP`    |
| UPTO 1M                  | `IndexIVFFlat` or `IndexHNSWFlat` |
| > 1M                      | `IndexIVFPQ` or `IndexHNSWFlat`   |


#### Document Class in langchain 

to attach extra context or info (the metadata) to the document alongside its text (page_content).

- The class Document extends a base named BaseMedia, which already has metadata: dict and an optional id field. 

- page_content is a required string field of Document class used to store the actual text. 

- Metadata is “arbitrary” — you can store whatever key/value pairs that make sense (e.g. “source”, “author”, “date”, “url”).

In [154]:
# from uuid import uuid4
from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]

In [155]:
index=faiss.IndexFlatIP(384)
vector_store=FAISS(
    embedding_function=embedding_model,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

In [156]:
vector_store.add_documents(documents=documents)

['6870e700-7ada-4f08-bf91-1005292aab21',
 'b92b7fd2-2001-4290-bfa9-9b8d70c315e9',
 'dec7e92f-bb40-4f34-a3fb-b833fc72f7c0',
 '4918bb75-9607-4367-9d93-f41c4b08099d',
 'c7a1a43d-5b8b-446a-b5a8-07878ef2b030',
 '5107740c-73f1-491a-9b79-e60b8db84287',
 '60adaea4-67eb-47fe-becc-0dd78504ed05',
 '47faf65e-f498-419f-94f5-415870c9e5dc',
 '08383839-dcd8-49cf-bea8-7590919ba643',
 'dd3c9734-04f4-4058-afbf-8c349161c94c']

In [157]:
# Give Top 2 to my Query
vector_store.similarity_search(
    "LangChain provides abstractions to make working with LLMs easy",
    k=2 #hyperparameter
)

[Document(id='dec7e92f-bb40-4f34-a3fb-b833fc72f7c0', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='47faf65e-f498-419f-94f5-415870c9e5dc', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!')]

#### Adding Filters

In [158]:
# Adding Filters 
vector_store.similarity_search(
    "LangChain provides abstractions to make working with LLMs easy",
    #k=2 #hyperparameter,
    filter={"source":{"$eq": "news"}} 
)

[Document(id='4918bb75-9607-4367-9d93-f41c4b08099d', metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.'),
 Document(id='b92b7fd2-2001-4290-bfa9-9b8d70c315e9', metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.'),
 Document(id='08383839-dcd8-49cf-bea8-7590919ba643', metadata={'source': 'news'}, page_content='The stock market is down 500 points today due to fears of a recession.')]

In [159]:
result=vector_store.similarity_search(
    "LangChain provides abstractions to make working with LLMs easy",
    #k=2, #hyperparameter,
    filter={"source":"news"}
)

print(result[0].metadata)

print(result[0].page_content)

{'source': 'news'}
Robbers broke into the city bank and stole $1 million in cash.


#### Retriever

In [160]:
retriever=vector_store.as_retriever(search_kwargs={"k": 3})

In [161]:
retriever.invoke("LangChain provides abstractions to make working with LLMs easy")

[Document(id='dec7e92f-bb40-4f34-a3fb-b833fc72f7c0', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='47faf65e-f498-419f-94f5-415870c9e5dc', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!'),
 Document(id='dd3c9734-04f4-4058-afbf-8c349161c94c', metadata={'source': 'tweet'}, page_content='I have a bad feeling I am going to get deleted :(')]

#### Saving Vector DB from Memory to Disk

- inmemory = RAM
- ondisk = SSD
- cloud(yet to discuss)

In [162]:
vector_store.save_local("today's class faiss index")

In [163]:
new_vector_store=FAISS.load_local(
  "today's class faiss index",embedding_model ,allow_dangerous_deserialization=True
)

In [164]:
new_vector_store.similarity_search("langchain")

[Document(id='dec7e92f-bb40-4f34-a3fb-b833fc72f7c0', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='47faf65e-f498-419f-94f5-415870c9e5dc', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!'),
 Document(id='c7a1a43d-5b8b-446a-b5a8-07878ef2b030', metadata={'source': 'tweet'}, page_content="Wow! That was an amazing movie. I can't wait to see it again."),
 Document(id='60adaea4-67eb-47fe-becc-0dd78504ed05', metadata={'source': 'website'}, page_content='The top 10 soccer players in the world right now.')]

In [165]:
from langchain_community.document_loaders import PyPDFLoader

In [166]:
FILE_PATH=r"/Users/tajamulkhan/Desktop/AgenticAI/2-Langchain/2.5-VectorDB/FAISS/llama2.pdf"

In [167]:
loader=PyPDFLoader(FILE_PATH)

In [168]:
len(loader.load())

77

In [169]:
pages=loader.load()

In [170]:
pages = []
async for page in loader.alazy_load():
    pages.append(page)

In [171]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [172]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,#hyperparameter
    chunk_overlap=50 #hyperparemeter
)

In [173]:
split_docs = splitter.split_documents(pages)

In [174]:
len(split_docs)

615

In [175]:
index=faiss.IndexFlatIP(384)
vector_store=FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

In [176]:
vector_store.add_documents(documents=split_docs)

['fca28a75-0bf6-4a0b-bf91-98f84e1d7b1c',
 '0996da48-4a57-4da7-aabe-1fbdf4784ed4',
 '27228630-5d3a-43ad-b9ee-d05ceea7c721',
 '86822409-a631-488a-9362-fa63e3d48b9c',
 'd0534ac3-fc34-4bf0-baf8-0bc14f43b125',
 'c7cdaeb4-dd4b-4579-8460-f841bf10899c',
 '3db84220-ee72-41a5-a622-5fac4ba39d42',
 '424967c2-cc81-432c-8f99-904777993908',
 '213b3c38-c30f-4d53-b5f9-169f3e5a8cf6',
 '3c7149c7-cf2e-44b2-9295-f71d2c15452a',
 '4c79f070-f4eb-4daa-93e8-80ccdf35328a',
 '35b27d79-f2e2-4c37-bc57-ad38d5c9ee02',
 '51d83193-dd36-4f9d-940b-31edaa4e23b5',
 '68f980b1-c1c8-4614-8341-a37135005809',
 'b8f5ca0d-ae49-475a-8e76-1650c670d865',
 'a2a78ef8-8205-438a-bd6c-a5b3390e2bbe',
 'b9d61215-292d-425d-9b48-16fed8a70000',
 'a380247b-677a-4b1e-a328-e1253c189901',
 '4d6b73ad-7e1d-4467-928f-5611d078fe8a',
 'b1f26447-73e9-4e0a-9477-bf5b0b4ac1ec',
 'd36d1d4a-cf8d-495f-8866-6638a35d72f0',
 '84a9b316-d0e6-459e-95c5-4f0c1fd40a91',
 '99a3ac9b-046a-412d-90a7-4f26b76aa8a7',
 '52f60e79-8715-48b7-809d-79ace3f800aa',
 '6fac007e-99ce-

In [177]:
retriever=vector_store.as_retriever(
    search_kwargs={"k": 10} #hyperparameter
)

In [178]:
retriever.invoke("what is llama model?")

[Document(id='6fac007e-99ce-4dcc-b831-a8f7525be4ef', metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '/Users/tajamulkhan/Desktop/AgenticAI/2-Langchain/2.5-VectorDB/FAISS/llama2.pdf', 'total_pages': 77, 'page': 3, 'page_label': '4'}, page_content='work (Section 6), and conclusions (Section 7).\n‡https://ai.meta.com/resources/models-and-libraries/llama/\n§We are delaying the release of the 34B model due to a lack of time to sufficiently red team.\n¶https://ai.meta.com/llama\n‖https://github.com/facebookresearch/llama\n4'),
 Document(id='0d3243a5-ccff-4133-9ba4-c032d096db32', metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+

In [179]:
from langchain_openai import ChatOpenAI
model=ChatOpenAI(temperature=7)

In [180]:
from langchain import hub
prompt = hub.pull("rlm/rag-prompt")

In [181]:
import pprint

In [182]:
pprint.pprint(prompt.messages)

[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})]


[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})]

In [183]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)
    

In [ ]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

In [ ]:
rag_chain.invoke("what is llama model?")

'Llama is a large language model developed by Meta.  It has various versions, including Llama 1 and Llama 2, with different parameter sizes.  Llama 2 is designed for commercial and research use and is available under a custom commercial license.'